## Section 1 (Locked): Environment Setup
Colab-first setup for M0 entropy observation. This notebook is observational only (no REAL control loop).

In [ ]:
# If needed in Colab, uncomment the next line:
# %pip install -q transformer-lens torch matplotlib numpy

import json
import random
import re
import subprocess
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from transformer_lens import HookedTransformer


def find_phase5_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "plan.md").exists() and (candidate / "README.md").exists():
            return candidate
        nested = candidate / "Phase 2" / "Phase 5"
        if (nested / "plan.md").exists():
            return nested
    raise FileNotFoundError("Could not locate Phase 5 root from current working directory.")


def sanitize_tag(value: str) -> str:
    return re.sub(r"[^a-z0-9]+", "_", value.lower()).strip("_")


PHASE5_ROOT = find_phase5_root(Path.cwd())
PROMPT_PATH = PHASE5_ROOT / "experiments" / "m0" / "prompts_m0.json"
RESULTS_ROOT = PHASE5_ROOT / "experiments" / "m0" / "results"

RUN_MODE = "smoke"  # "smoke" or "full"

# Hot-swappable model registry (choose MODEL_KEY, optionally add fallback keys)
MODEL_NAME_MAP = {
    "qwen3_0_6b": "Qwen/Qwen3-0.6B",
    "qwen2_5_0_5b": "Qwen/Qwen2.5-0.5B",
    "tinyllama_1_1b": "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    "pythia_1b": "EleutherAI/pythia-1b",
    "pythia_2_8b": "EleutherAI/pythia-2.8b",
}
MODEL_KEY = "qwen3_0_6b"
FALLBACK_MODEL_KEYS = ["qwen2_5_0_5b"]

if MODEL_KEY not in MODEL_NAME_MAP:
    raise ValueError(f"Unknown MODEL_KEY: {MODEL_KEY}. Available: {sorted(MODEL_NAME_MAP)}")
for key in FALLBACK_MODEL_KEYS:
    if key not in MODEL_NAME_MAP:
        raise ValueError(f"Unknown fallback key: {key}")

REQUESTED_MODEL_KEYS = [MODEL_KEY] + [k for k in FALLBACK_MODEL_KEYS if k != MODEL_KEY]
REQUESTED_MODEL_NAMES = [MODEL_NAME_MAP[k] for k in REQUESTED_MODEL_KEYS]

SEED = 42
TEMPERATURE = 0.8
SMOKE_MAX_NEW_TOKENS = 64
FULL_MAX_NEW_TOKENS = 128
BOOTSTRAP_SAMPLES = 3000
PERMUTATION_SAMPLES = 3000

PRIMARY_METRICS = [
    "final_layer_token_entropy_mean",
    "token_entropy_volatility_std",
]
SECONDARY_METRICS = [
    "per_head_attention_entropy_mean",
    "hidden_state_delta_norm_mean",
]

MAX_NEW_TOKENS = SMOKE_MAX_NEW_TOKENS if RUN_MODE == "smoke" else FULL_MAX_NEW_TOKENS
SAMPLES_PER_CLASS = 2 if RUN_MODE == "smoke" else 12

EXPERIMENT_TAG = f"{sanitize_tag(MODEL_KEY)}_{sanitize_tag(RUN_MODE)}"
RESULTS_DIR = RESULTS_ROOT / EXPERIMENT_TAG
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32

print(f"Phase 5 root: {PHASE5_ROOT}")
print(f"Prompt path:  {PROMPT_PATH}")
print(f"Results root: {RESULTS_ROOT}")
print(f"Results dir:  {RESULTS_DIR}")
print(f"Device: {device} | dtype: {dtype}")
print(f"Run mode: {RUN_MODE} | prompts per class: {SAMPLES_PER_CLASS} | max_new_tokens: {MAX_NEW_TOKENS}")
print(f"Requested model keys:  {REQUESTED_MODEL_KEYS}")
print(f"Requested model names: {REQUESTED_MODEL_NAMES}")


## Section 2 (Locked): Prompt Bank Load and Class Labels
Loads paired prompts and enforces stable IDs/class balance.

In [ ]:
with PROMPT_PATH.open("r", encoding="utf-8") as f:
    prompt_bank = json.load(f)

prompts = prompt_bank["prompts"]
non_competing = [p for p in prompts if p["class"] == "non_competing"]
competing = [p for p in prompts if p["class"] == "competing"]

assert len(non_competing) >= 12, "Expected at least 12 non_competing prompts"
assert len(competing) >= 12, "Expected at least 12 competing prompts"
assert all(p["id"].startswith("nc_") for p in non_competing)
assert all(p["id"].startswith("cp_") for p in competing)

selected_prompts = non_competing[:SAMPLES_PER_CLASS] + competing[:SAMPLES_PER_CLASS]
print(f"Loaded {len(prompts)} prompts total.")
print(f"Selected {len(non_competing[:SAMPLES_PER_CLASS])} non_competing and {len(competing[:SAMPLES_PER_CLASS])} competing prompts for this run.")

selected_prompts[:2]

## Section 3 (Locked): Generation and Hook Capture
Runs token generation while capturing final-layer token entropy, attention entropy, and residual trajectory snapshots.

In [ ]:
def try_load_model(model_keys: list[str]):
    errors = []
    for model_key in model_keys:
        model_name = MODEL_NAME_MAP[model_key]
        try:
            print(f"Loading model [{model_key}]: {model_name}")
            model = HookedTransformer.from_pretrained(
                model_name,
                device=device,
                dtype=dtype,
            )
            print(f"Loaded model [{model_key}]: {model_name}")
            return model, model_key, model_name
        except Exception as exc:
            errors.append({"model_key": model_key, "model_name": model_name, "error": str(exc)})
            print(f"Failed model [{model_key}]: {exc}")
    raise RuntimeError(f"Could not load any requested model. Errors: {errors}")


def shannon_entropy(prob_tensor: torch.Tensor) -> torch.Tensor:
    p = prob_tensor.clamp_min(1e-12)
    return -(p * torch.log(p)).sum(dim=-1)


@torch.no_grad()
def run_prompt_capture(model: HookedTransformer, prompt_text: str, max_new_tokens: int, temperature: float):
    tokens = model.to_tokens(prompt_text, prepend_bos=True).to(device)

    token_entropies = []
    attention_entropies = []
    residual_snapshots = []
    generated_token_ids = []

    final_layer = model.cfg.n_layers - 1

    for _ in range(max_new_tokens):
        logits, cache = model.run_with_cache(tokens, remove_batch_dim=False)
        last_logits = logits[:, -1, :].float()
        scaled_logits = last_logits / max(temperature, 1e-6)
        probs = torch.softmax(scaled_logits, dim=-1)

        step_entropy = shannon_entropy(probs)
        token_entropies.append(float(step_entropy.item()))

        # Attention tensor capture (final layer, per-head entropy at query=-1)
        pattern = cache["pattern", final_layer][0, :, -1, :]  # [heads, key_positions]
        head_entropy = shannon_entropy(pattern)
        attention_entropies.append(float(head_entropy.mean().item()))

        # Residual snapshot capture (final layer, last token)
        resid = cache["resid_post", final_layer][0, -1, :].float().detach().cpu()
        residual_snapshots.append(resid)

        next_token = torch.multinomial(probs, num_samples=1)
        next_token_id = int(next_token.item())
        generated_token_ids.append(next_token_id)
        tokens = torch.cat([tokens, next_token.to(tokens.device)], dim=1)

        eos_id = model.tokenizer.eos_token_id
        if eos_id is not None and next_token_id == eos_id:
            break

    if len(residual_snapshots) >= 2:
        deltas = [
            float(torch.norm(residual_snapshots[i] - residual_snapshots[i - 1], p=2).item())
            for i in range(1, len(residual_snapshots))
        ]
    else:
        deltas = [0.0]

    generated_text = model.to_string(generated_token_ids) if generated_token_ids else ""

    metrics = {
        "final_layer_token_entropy_mean": float(np.mean(token_entropies)) if token_entropies else 0.0,
        "token_entropy_volatility_std": float(np.std(token_entropies)) if token_entropies else 0.0,
        "per_head_attention_entropy_mean": float(np.mean(attention_entropies)) if attention_entropies else 0.0,
        "hidden_state_delta_norm_mean": float(np.mean(deltas)) if deltas else 0.0,
        "num_generated_tokens": int(len(generated_token_ids)),
    }

    traces = {
        "token_entropies": token_entropies,
        "attention_entropies": attention_entropies,
        "hidden_state_delta_norms": deltas,
    }

    return metrics, traces, generated_text


model, resolved_model_key, resolved_model_name = try_load_model(REQUESTED_MODEL_KEYS)


## Section 4 (Locked): Metric Extraction
Runs all selected prompts and extracts locked M0 metrics (primary + secondary).

In [ ]:
records = []
for prompt_obj in selected_prompts:
    metrics, traces, generated_text = run_prompt_capture(
        model=model,
        prompt_text=prompt_obj["prompt"],
        max_new_tokens=MAX_NEW_TOKENS,
        temperature=TEMPERATURE,
    )

    record = {
        "id": prompt_obj["id"],
        "class": prompt_obj["class"],
        "topic": prompt_obj["topic"],
        "prompt": prompt_obj["prompt"],
        "generated_text_preview": generated_text[:240],
        **metrics,
        "traces": traces,
    }
    records.append(record)

print(f"Completed metric extraction for {len(records)} prompts.")
records[0]

## Section 5 (Locked): Statistical Comparison and Plots
Computes bootstrap CI, Cliff delta, and permutation p-values for competing-minus-non_competing differences.

In [ ]:
def cliffs_delta(x: np.ndarray, y: np.ndarray) -> float:
    gt = 0
    lt = 0
    for xv in x:
        gt += np.sum(xv > y)
        lt += np.sum(xv < y)
    n = len(x) * len(y)
    return float((gt - lt) / n) if n else 0.0


def bootstrap_mean_diff_ci(x: np.ndarray, y: np.ndarray, n_boot: int, seed: int):
    rng = np.random.default_rng(seed)
    diffs = []
    for _ in range(n_boot):
        xb = rng.choice(x, size=len(x), replace=True)
        yb = rng.choice(y, size=len(y), replace=True)
        diffs.append(float(np.mean(xb) - np.mean(yb)))
    low, high = np.percentile(diffs, [2.5, 97.5])
    return float(low), float(high)


def permutation_pvalue_greater(x: np.ndarray, y: np.ndarray, n_perm: int, seed: int):
    rng = np.random.default_rng(seed)
    observed = float(np.mean(x) - np.mean(y))
    combined = np.concatenate([x, y])
    x_n = len(x)
    count = 0
    for _ in range(n_perm):
        shuffled = rng.permutation(combined)
        diff = float(np.mean(shuffled[:x_n]) - np.mean(shuffled[x_n:]))
        if diff >= observed:
            count += 1
    p_value = float((count + 1) / (n_perm + 1))
    return p_value, observed


def metric_values(metric_name: str, cls: str):
    return np.array([r[metric_name] for r in records if r["class"] == cls], dtype=float)


summary_by_metric = {}
for metric_name in PRIMARY_METRICS + SECONDARY_METRICS:
    c_vals = metric_values(metric_name, "competing")
    n_vals = metric_values(metric_name, "non_competing")

    ci_low, ci_high = bootstrap_mean_diff_ci(c_vals, n_vals, BOOTSTRAP_SAMPLES, SEED)
    p_value, observed = permutation_pvalue_greater(c_vals, n_vals, PERMUTATION_SAMPLES, SEED)
    delta = cliffs_delta(c_vals, n_vals)

    summary_by_metric[metric_name] = {
        "competing_mean": float(np.mean(c_vals)),
        "non_competing_mean": float(np.mean(n_vals)),
        "mean_diff_competing_minus_non_competing": observed,
        "bootstrap_95_ci": [ci_low, ci_high],
        "cliffs_delta": float(delta),
        "permutation_pvalue_greater": p_value,
    }

primary_positive = all(
    summary_by_metric[m]["mean_diff_competing_minus_non_competing"] > 0 for m in PRIMARY_METRICS
)
primary_ci_excludes_zero_any = any(
    summary_by_metric[m]["bootstrap_95_ci"][0] > 0 or summary_by_metric[m]["bootstrap_95_ci"][1] < 0
    for m in PRIMARY_METRICS
)
primary_effect_size_any = any(
    abs(summary_by_metric[m]["cliffs_delta"]) > 0.2 for m in PRIMARY_METRICS
)

directional_support = bool(primary_positive and primary_ci_excludes_zero_any and primary_effect_size_any)

acceptance = {
    "both_primary_differences_positive": primary_positive,
    "bootstrap_ci_excludes_zero_for_at_least_one_primary": primary_ci_excludes_zero_any,
    "cliffs_delta_gt_0_2_for_at_least_one_primary": primary_effect_size_any,
    "directional_support_at_selected_model": directional_support,
}

if directional_support:
    failure_handling_note = f"Directional support observed at {resolved_model_name}."
    next_decision = "Proceed to M1 adapter implementation decision."
else:
    failure_handling_note = f"No directional support at {resolved_model_name}."
    next_decision = "Queue TinyLlama replication as next decision (not automatic in M0)."

summary_stats = {
    "experiment_tag": EXPERIMENT_TAG,
    "model_key": resolved_model_key,
    "model_name": resolved_model_name,
    "run_mode": RUN_MODE,
    "samples_per_class": SAMPLES_PER_CLASS,
    "primary_metrics": PRIMARY_METRICS,
    "secondary_metrics": SECONDARY_METRICS,
    "summary_by_metric": summary_by_metric,
    "acceptance": acceptance,
    "failure_handling_note": failure_handling_note,
    "next_decision": next_decision,
}

print(json.dumps(summary_stats["acceptance"], indent=2))
print(summary_stats["failure_handling_note"])
print(summary_stats["next_decision"])

# Plot primary metric distributions
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for idx, metric_name in enumerate(PRIMARY_METRICS):
    c_vals = metric_values(metric_name, "competing")
    n_vals = metric_values(metric_name, "non_competing")
    axes[idx].boxplot([n_vals, c_vals], labels=["non_competing", "competing"])
    axes[idx].set_title(metric_name)
    axes[idx].set_ylabel("value")
    axes[idx].grid(alpha=0.3)

plt.tight_layout()
plt.show()


## Section 6 (Locked): Artifact Export
Writes raw prompt metrics, aggregate stats, and run metadata JSON artifacts.

In [ ]:
def git_commit_hash() -> str | None:
    try:
        out = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True)
        return out.strip()
    except Exception:
        return None


run_meta = {
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "phase5_root": str(PHASE5_ROOT),
    "prompt_path": str(PROMPT_PATH),
    "results_root": str(RESULTS_ROOT),
    "results_dir": str(RESULTS_DIR),
    "experiment_tag": EXPERIMENT_TAG,
    "run_mode": RUN_MODE,
    "requested_model_keys": REQUESTED_MODEL_KEYS,
    "requested_model_names": REQUESTED_MODEL_NAMES,
    "resolved_model_key": resolved_model_key,
    "resolved_model_name": resolved_model_name,
    "seed": SEED,
    "temperature": TEMPERATURE,
    "max_new_tokens": MAX_NEW_TOKENS,
    "samples_per_class": SAMPLES_PER_CLASS,
    "git_commit": git_commit_hash(),
}

metrics_path = RESULTS_DIR / "metrics_raw.jsonl"
summary_path = RESULTS_DIR / "summary_stats.json"
meta_path = RESULTS_DIR / "run_meta.json"

with metrics_path.open("w", encoding="utf-8") as f:
    for row in records:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

with summary_path.open("w", encoding="utf-8") as f:
    json.dump(summary_stats, f, indent=2)

with meta_path.open("w", encoding="utf-8") as f:
    json.dump(run_meta, f, indent=2)

# Lightweight schema checks
required_raw_keys = {
    "id", "class", "topic", "prompt",
    "final_layer_token_entropy_mean",
    "token_entropy_volatility_std",
    "per_head_attention_entropy_mean",
    "hidden_state_delta_norm_mean",
    "num_generated_tokens",
    "traces",
}
for row in records:
    missing = required_raw_keys - set(row.keys())
    assert not missing, f"Raw record missing keys: {missing}"

assert "summary_by_metric" in summary_stats
assert "acceptance" in summary_stats
assert "resolved_model_name" in run_meta
assert "resolved_model_key" in run_meta
assert "experiment_tag" in run_meta

print(f"Wrote raw metrics: {metrics_path}")
print(f"Wrote summary:     {summary_path}")
print(f"Wrote run meta:    {meta_path}")

# Quick smoke-level visibility
print("\nAcceptance snapshot:")
print(json.dumps(summary_stats["acceptance"], indent=2))
